# Адресный native-повтор: выбрать lr по валидации
Два заранее заданныхlr .02и.1 сравниваются только на апреле–июне. Рецепт и число деревьев сохраняю до июльского final fit. Остальные13обучений прошлого прохода не повторяю. Данные знакомы; это проверка исправленного порядка, не независимое измерение обучения.


In [1]:
import os,time,json,sys
from pathlib import Path
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ[name]='1'
import numpy as np,pandas as pd,lightgbm as lgb,sklearn
from sklearn.metrics import mean_squared_error
RANDOM_STATE=0
print(sys.version.split()[0],pd.__version__,sklearn.__version__,lgb.__version__)
s=pd.read_csv('../project/data/daily_sales.csv',parse_dates=['date']).sort_values(['shop_id','date']).reset_index(drop=True)
CAT=['shop_id','format']
fit=s.date<'2025-04-01';val=(s.date>='2025-04-01')&(s.date<'2025-07-01');refit=s.date<'2025-07-01';test=~refit
def prep(source,as_of):
 a=source.copy();h=a[a.date<as_of];future=a.date>=as_of
 for c in ['temp','rain']:
  a[c]=a[c].astype(float);monthly=h.groupby(h.date.dt.month)[c].mean()
  a.loc[future,c]=a.loc[future,'date'].dt.month.map(monthly)
 a['weekday']=a.date.dt.weekday;a['month']=a.date.dt.month;a['weekend']=(a.weekday>=5).astype(int);a['day_n']=(a.date-pd.Timestamp('2024-01-01')).dt.days
 for c in CAT:a[c]=pd.Categorical(a[c],categories=sorted(h[c].unique()))
 return a[['temp','rain','holiday','promo','weekday','month','weekend','day_n','shop_id','format']]
y=s.revenue_k;dev=prep(s,'2025-04-01')
def rmse(y,p):return float(np.sqrt(mean_squared_error(y,p)))
print(Path('protocol-before-fit.json').read_text())


3.12.14 3.0.6 1.8.0 4.7.0
{
  "candidate_rates": [
    0.02,
    0.1
  ],
  "model": "LightGBM",
  "leaves": 31,
  "min_child_samples": 20,
  "seed": 0,
  "cap": 3000,
  "patience": 100,
  "threads": 1,
  "fit": "before2025-04-01",
  "selection": "minimum Apr-Jun RMSE using April-issued scenario",
  "final": "only chosen recipe, refit beforeJuly, scoreJuly-Dec once",
  "retest_limitation": "Known data, no claim of new independent assessment"
}


In [2]:
rows=[]
for lr in [.02,.1]:
 probe=lgb.LGBMRegressor(n_estimators=3000,learning_rate=lr,num_leaves=31,min_child_samples=20,random_state=0,n_jobs=1,verbose=-1)
 start=time.perf_counter();probe.fit(dev[fit],y[fit],eval_X=dev[val],eval_y=y[val],callbacks=[lgb.early_stopping(100,verbose=False)]);search_s=time.perf_counter()-start
 rows.append({'lr':lr,'trees':int(probe.best_iteration_),'val_rmse_k':rmse(y[val],probe.predict(dev[val])),'selection_fit_s':search_s})
comparison=pd.DataFrame(rows);chosen=min(rows,key=lambda r:r['val_rmse_k'])
Path('selection-before-final.json').write_text(json.dumps({'as_of':'2025-04-01','candidates':rows,'chosen':chosen,'metric':'Apr-Jun RMSE, thousand RUB per shop-day'},indent=2))
comparison.to_csv('validation.csv',index=False)
print(comparison.to_string(index=False));print('chosen before final:',chosen)


  lr  trees  val_rmse_k  selection_fit_s
0.02    325   11.683124         0.070289
0.10    595   11.376204         0.109640
chosen before final: {'lr': 0.1, 'trees': 595, 'val_rmse_k': 11.376203822092641, 'selection_fit_s': 0.10963983298279345}


In [3]:
X=prep(s,'2025-07-01')
m=lgb.LGBMRegressor(n_estimators=chosen['trees'],learning_rate=chosen['lr'],num_leaves=31,min_child_samples=20,random_state=0,n_jobs=1,verbose=-1)
start=time.perf_counter();m.fit(X[refit],y[refit]);fit_s=time.perf_counter()-start
start=time.perf_counter();p=m.predict(X[test]);predict_ms=(time.perf_counter()-start)*1000
result={'as_of':'2025-07-01','rows':int(test.sum()),'lr':chosen['lr'],'trees':chosen['trees'],'final_rmse_k':rmse(y[test],p),'final_fit_s':fit_s,'predict_ms':predict_ms}
Path('final.json').write_text(json.dumps(result,indent=2));print(result)


{'as_of': '2025-07-01', 'rows': 1104, 'lr': 0.1, 'trees': 595, 'final_rmse_k': 13.203625546537204, 'final_fit_s': 0.08858812501421198, 'predict_ms': 14.154667034745216}


**Вывод по фактическому повтору.** Валидационный RMSE дляlr0.02 —11.683тыс.₽ (325деревьев), для0.1 —11.376 (595деревьев). До финального обучения выбран0.1 и595деревьев; selection-before-final.json записан раньше final.json.
После переобучения доиюля финальный RMSE13.204тыс.₽ на1104строках, fit0.0886с и пакетный predict14.15мс в этом запуске. Финальная ошибка не повод заменить выбранное правило задним числом; новый кандидат потребует следующего независимого периода. Это адресный повтор знакомого набора.
